In [2]:
import os
import zipfile
from pathlib import Path

# 프로젝트 루트 디렉터리
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"데이터 디렉터리: {DATA_DIR}")

# data 디렉터리 생성
DATA_DIR.mkdir(exist_ok=True)

프로젝트 루트: /home/wsm/workspace/hit-archlens-project
데이터 디렉터리: /home/wsm/workspace/hit-archlens-project/data


## 방법 1: wget을 사용한 다운로드

터미널에서 다음 명령어를 실행하세요:


In [3]:
# wget을 사용한 다운로드 예시 (주석 처리)
# !wget https://d1.awsstatic.com/webteam/architecture-icons/q1-2024/Asset-Package_01242024.7c4f8b8b.zip -O data/Asset-Package.zip

# 또는 최신 버전 URL 확인 후 다운로드
# AWS 공식 사이트에서 최신 다운로드 링크를 확인하세요

## 방법 2: Python을 사용한 다운로드


In [4]:
import requests, zipfile
from tqdm import tqdm

def ensure_zipfile(zip_path):
    # zip 파일이 존재하고, 충분히 크고, zip포맷인지 체크. 아니면 False 반환
    if not zip_path.exists():
        print(f"❌ 파일이 없습니다: {zip_path}")
        return False
    size = zip_path.stat().st_size
    if size < 1024:
        print(f"⚠️ 크기가 너무 작음({size}B), 다운로드가 HTML일 수 있습니다.")
        return False
    if not zipfile.is_zipfile(zip_path):
        print(f"⚠️ ZIP 포맷이 아닙니다: {zip_path}")
        return False
    return True

# 간단한 파일 다운로드 함수. tqdm으로 진행 표시, zip체크 옵션
def download_file(url, output_path, chunk_size=8192, validate_zip=True):
    resp = requests.get(url, stream=True, timeout=30)
    resp.raise_for_status()
    total = int(resp.headers.get('content-length', 0))
    with open(output_path, "wb") as f, tqdm(total=total, unit='B', unit_scale=True, desc=output_path.name) as bar:
        for chunk in resp.iter_content(chunk_size):
            if chunk:
                f.write(chunk)
                bar.update(len(chunk))
    if validate_zip and not ensure_zipfile(output_path):
        raise ValueError("다운로드된 파일이 ZIP이 아닙니다.")
    print(f"✅ 완료: {output_path}")
    return output_path

# 예시 (실행 전 경로/URL 확인)
# url = "https://d1.awsstatic.com/webteam/architecture-icons/q1-2024/Asset-Package_01242024.7c4f8b8b.zip"
# download_file(url, DATA_DIR / "Asset-Package.zip")

## 압축 해제


In [5]:
# ZIP 파일 자동 압축 해제 (간단버전)

for zip_file in DATA_DIR.glob("Asset-Package*.zip"):
    print(f"\n{zip_file.name}")
    if not ensure_zipfile(zip_file):
        print("  ⚠️ 유효하지 않은 ZIP (HTML/손상) — 스킵")
        continue

    extract_dir = DATA_DIR / zip_file.stem
    if extract_dir.exists():
        print(f"  이미 압축 해제됨: {extract_dir}")
        continue

    try:
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print(f"  ✅ 압축 해제 완료 → {DATA_DIR}")
    except Exception as e:
        print(f"  ⚠️ 압축 해제 실패: {e}")

# ZIP 파일이 없는 경우 안내
if not list(DATA_DIR.glob("Asset-Package*.zip")):
    print("⚠️  ZIP 파일을 찾을 수 없습니다. 먼저 다운로드하세요.")


Asset-Package.zip
⚠️ 크기가 너무 작음(111B), 다운로드가 HTML일 수 있습니다.
  ⚠️ 유효하지 않은 ZIP (HTML/손상) — 스킵


## 압축 해제 결과 확인


In [6]:
# 압축 해제된 디렉터리 확인
asset_dirs = list(DATA_DIR.glob("Asset-Package*"))
asset_dirs = [d for d in asset_dirs if d.is_dir()]

if asset_dirs:
    print("📁 압축 해제된 디렉터리:")
    for asset_dir in asset_dirs:
        print(f"\n{asset_dir.name}/")
        subdirs = [d for d in asset_dir.iterdir() if d.is_dir()]
        for subdir in subdirs:
            file_count = len(list(subdir.rglob("*.png"))) + len(list(subdir.rglob("*.svg")))
            print(f"  ├── {subdir.name}/ ({file_count}개 파일)")
else:
    print("⚠️  압축 해제된 디렉터리를 찾을 수 없습니다.")

📁 압축 해제된 디렉터리:

Asset-Package_02072025.dee42cd0a6eaacc3da1ad9519579357fb546f803/
  ├── Architecture-Group-Icons_02072025/ (30개 파일)
  ├── Architecture-Service-Icons_02072025/ (2781개 파일)
  ├── Resource-Icons_02072025/ (1054개 파일)
  ├── __MACOSX/ (4020개 파일)
  ├── Category-Icons_02072025/ (234개 파일)
